In [7]:
# ---- Imports ----
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import matplotlib.pyplot as plt
from scipy import stats

# ---- Paths ----
PROJECT_ROOT = Path().resolve().parent.parent   # adjust if needed
raw_path = PROJECT_ROOT / "data" / "Oil&GasWells.csv"

print("Loading:", raw_path)

# ---- Load Raw Data ----
df = pd.read_csv(raw_path)
print("Loaded dataset:", df.shape)


# ============================================
# SECTION 1 — CLEANING COLUMNS
# ============================================

# ---- Convert date columns ----
date_cols = [
    'Well_Date_Approved',
    'Well_Date_Complete',
    'Well_Date_Plugged',
    'Last_Inspection_Date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')


# ---- Clean coordinates ----
# Keep only surface coordinates (other coordinate systems unusable)
df = df.dropna(subset=['Well Latitude', 'Well Longitude']).copy()

# Convert to radians for haversine distances later
df['lat_rad'] = np.radians(df['Well Latitude'])
df['lon_rad'] = np.radians(df['Well Longitude'])


# ---- Drop columns with >99% NA ----
na_pct = df.isna().mean()
cols_to_drop = na_pct[na_pct > 0.99].index.tolist()

df = df.drop(columns=cols_to_drop)
print("Dropped sparse columns:", cols_to_drop)


# ---- Engineered Features ----

# Horizontal approximation using surface vs bottom hole delta
df['lat_delta'] = abs(df['Well Latitude'] - df.get('Bottom Hole Latitude', 0))
df['is_horizontal'] = (df['lat_delta'] > 0.005).astype(int)

# Well age
df['years_since_approved'] = (
    pd.Timestamp.today().year - df['Well_Date_Approved'].dt.year
)

df['age_category'] = pd.cut(
    df['years_since_approved'],
    bins=[0,5,10,20,40,100],
    labels=['0–5','6–10','11–20','21–40','40+']
)

Loading: /Users/tylergreenwood/Documents/GitHub/data-quarks/data/Oil&GasWells.csv
Loaded dataset: (24174, 58)
Dropped sparse columns: ['Toe Longitude', 'Toe Latitude', 'OWPS Description']


In [8]:

# ---- Identify injection wells ----
injection_mask = df['Well Type'].str.contains('Injection', case=False, na=False)
injection_wells = df[injection_mask].copy()

# ---- Identify production wells ----
production_keywords = ['oil', 'gas', 'producing', 'plugged', 'final restoration']
production_mask = (
    df['Well Status'].str.contains('|'.join(production_keywords), case=False, na=False) |
    df['Well Type'].str.contains('oil|gas', case=False, na=False)
)

production_wells = df[production_mask].copy()

print(f"Injection wells: {len(injection_wells):,}")
print(f"Production wells: {len(production_wells):,}")


# ---- BallTree for nearest neighbors ----
coords_prod = production_wells[['lat_rad', 'lon_rad']].values
tree = BallTree(coords_prod, metric='haversine')

R = 3958.8  # Earth's radius in miles
radius_miles = 7.0
radius_radians = radius_miles / R

links = []

for idx, inj in injection_wells.iterrows():
    inj_coord = np.array([[inj['lat_rad'], inj['lon_rad']]])
    indices = tree.query_radius(inj_coord, r=radius_radians)[0]

    if len(indices) == 0:
        continue

    lat1, lon1 = inj['lat_rad'], inj['lon_rad']
    lat2 = production_wells.iloc[indices]['lat_rad'].values
    lon2 = production_wells.iloc[indices]['lon_rad'].values

    # Haversine distance
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distances = R * c

    for prod_idx, dist in zip(indices, distances):
        prod = production_wells.iloc[prod_idx]
        links.append({
            "injection_OBJ_ID": inj["OBJ_ID"],
            "injection_API": inj["Permit number - API"],
            "injection_lat": inj["Well Latitude"],
            "injection_lon": inj["Well Longitude"],

            "production_OBJ_ID": prod["OBJ_ID"],
            "production_API": prod["Permit number - API"],
            "production_lat": prod["Well Latitude"],
            "production_lon": prod["Well Longitude"],

            "distance_miles": dist
        })

link_df = pd.DataFrame(links)
print("Link table size:", link_df.shape)


# ============================================
# SECTION 3 — PRODUCTION CHANGE METRIC
# ============================================

# EXAMPLE: Simple proxy production change metric
# Modify if you have year-by-year production
if "Last_Nonzero_Production_Year" in df.columns:
    df['prod_change'] = df['Last_Nonzero_Production_Year'].diff().fillna(0)
else:
    df['prod_change'] = np.random.normal(0, 10, len(df))  # placeholder if missing

# Merge prod_change into link_df
link_df = link_df.merge(
    df[['OBJ_ID','prod_change']],
    left_on='production_OBJ_ID',
    right_on='OBJ_ID',
    how='left'
)


# ============================================
# SECTION 4 — DISTANCE BANDS
# ============================================

bins = [0, 1, 3, 5, 10, np.inf]
labels = ["0–1", "1–3", "3–5", "5–10", "10+"]

link_df['distance_band'] = pd.cut(link_df['distance_miles'], bins=bins, labels=labels)

link_df.to_csv(PROJECT_ROOT / "data" / "link_table.csv", index=False)
df.to_csv(PROJECT_ROOT / "data" / "cleaned_wells.csv", index=False)


Injection wells: 83
Production wells: 20,738
Link table size: (15527, 9)
